# PCAWG trinucleotide mutation probabilities

This notebook calculates trinucleotide-context mutation probabilities for the PCAWG Liver-HCC cohort.

Mutation probabilities are calculated separately for the complete cohort and for subsets used in the outlier analyses:

1. All 314 Liver-HCC samples
2. Cohort excluding mutation-count outliers
3. Mutation-count outliers only
4. Cohort excluding the 13 NFIA-specific outliers
5. The 13 NFIA-specific outliers only
6. Cohort excluding the top 3 NFIA-specific outliers
7. The top 3 NFIA-specific outliers only

For each group, the observed counts of the 192 possible trinucleotide substitutions are normalised by the genomic frequency of the corresponding reference trinucleotide. The resulting values are then normalised to obtain the mutation-probability distribution used to calculate expected mutation counts in downstream analyses.


In [2]:
## libraries
from pathlib import Path
import numpy as np
import pandas as pd
import bgreference

In [3]:

def find_repo_root():
    "'repository root'"
    current = Path.cwd().resolve()

    for path in [current, *current.parents]:
        if (path / "notebooks").exists() and (path / "results").exists():
            return path

    raise FileNotFoundError("Could not locate repository root.")

REPO_ROOT = find_repo_root()

In [ ]:
TRI_FREQ_FILE = REPO_ROOT / "data" / "derived" / "reference" / "hg19_trinucleotide_counts.tsv"

PCAWG_MUTATION_FILE = REPO_ROOT / "data" / "raw" / "pcawg" / "PCAWG_Liver-HCC_SBS.bed.gz"

OUTPUT_DIR = REPO_ROOT / "results" / "preprocessing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#### Genome-wide frequencies of the 64 trinucleotide contexts 

Calculated from the hg19 reference genome

In [3]:
tri_freq_df = pd.read_csv(TRI_FREQ_FILE, sep="\t",header=None, names=["tri","count"])

print("Number of trinucleotides:", len(tri_freq_df))
tri_freq_df.head()

Number of trinucleotides: 64


,tri,count
0,CGA,2853396
1,TCG,2859169
2,CGC,3100790
3,GCG,3105472
4,ACG,3607157


#### Mutation File - PCAWG Liver HCC SBS mutations

Somatic single-base substitution (SBS) from 314 PCAWG Liver-HCC samples were used for the analysis (hg19)

In [5]:
HCC = pd.read_csv(PCAWG_MUTATION_FILE, sep="\t", header=None)

HCC.rename(columns={0:"chr", 1:"start", 2:"end", 3:"ref", 4:"alt", 5:"sampleID", 6:"Type"},
    inplace=True)

print(f"Number of mutations: {len(HCC):,}")
print(f"Number of samples: {HCC['sampleID'].nunique()}")

HCC.head()

Number of mutations: 3,845,119
Number of samples: 314


,chr,start,end,ref,alt,sampleID,Type
0,chr1,10505,10506,C,A,4cff8590-559e-4204-8635-96e11bfeda68,Liver-HCC
1,chr1,20251,20252,A,G,45aedf22-c622-11e3-bf01-24c6515278c0,Liver-HCC
2,chr1,77903,77904,A,G,e34bcc5e-c622-11e3-bf01-24c6515278c0,Liver-HCC
3,chr1,94993,94994,T,C,221d0290-c623-11e3-bf01-24c6515278c0,Liver-HCC
4,chr1,121080,121081,T,C,19fe27d8-c623-11e3-bf01-24c6515278c0,Liver-HCC


In [6]:
# counting total mutations in each sample
samples = HCC["sampleID"].value_counts()
samples_pcawg = pd.DataFrame({'sampleID': samples.index, 'Count': samples.values})
samples_pcawg["Type"] = "Liver-HCC"
samples_pcawg.head()

,sampleID,Count,Type
0,96517058-18d2-4847-8085-768ca8aa1565,64258,Liver-HCC
1,c6bb32f0-c622-11e3-bf01-24c6515278c0,44433,Liver-HCC
2,468083cb-95c3-47c1-9d96-79bcf7bf3f26,36673,Liver-HCC
3,1295947c-c623-11e3-bf01-24c6515278c0,32597,Liver-HCC
4,63fef3fe-c622-11e3-bf01-24c6515278c0,31611,Liver-HCC


#### Outlier samples

In [8]:
#finidng out outliers using IQR method
# Q1 and Q3
Q1 = np.percentile(samples_pcawg["Count"], 25)
Q3 = np.percentile(samples_pcawg["Count"], 75)

print(f"Q1: {Q1}")
print(f"Q3: {Q3}")

Q1: 7940.75
Q3: 15791.5


In [9]:
#IQR
IQR = Q3 - Q1
print(f"IQR: {IQR}")

# Calculate the lower lower and upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Lower Bound: {lower_bound}")
print(f"Upper Bound: {upper_bound}")

IQR: 7850.75
Lower Bound: -3835.375
Upper Bound: 27567.625


In [10]:
sample_outliers = samples_pcawg[
    (samples_pcawg["Count"] < lower_bound) |
    (samples_pcawg["Count"] > upper_bound)].copy()

print("Number of mutation-count outliers:", len(sample_outliers))
sample_outliers

Number of mutation-count outliers: 8


,sampleID,Count,Type
0,96517058-18d2-4847-8085-768ca8aa1565,64258,Liver-HCC
1,c6bb32f0-c622-11e3-bf01-24c6515278c0,44433,Liver-HCC
2,468083cb-95c3-47c1-9d96-79bcf7bf3f26,36673,Liver-HCC
3,1295947c-c623-11e3-bf01-24c6515278c0,32597,Liver-HCC
4,63fef3fe-c622-11e3-bf01-24c6515278c0,31611,Liver-HCC
5,7fc7b316-f3d3-4e71-9417-423fe334e6bc,30355,Liver-HCC
6,98d27916-c622-11e3-bf01-24c6515278c0,28966,Liver-HCC
7,532259b8-c622-11e3-bf01-24c6515278c0,28754,Liver-HCC


In [11]:
HCC_wo_outliers = HCC[
    ~HCC["sampleID"].isin(sample_outliers["sampleID"])].copy()

HCC_outliers = HCC[
    HCC["sampleID"].isin(sample_outliers["sampleID"])].copy()

print("Samples excluding outliers:", HCC_wo_outliers["sampleID"].nunique())
print("Outlier samples:", HCC_outliers["sampleID"].nunique())

Samples excluding outliers: 306
Outlier samples: 8


#### Creating a df with and without NFIA outliers

In [48]:
nfia_outliers = (['cf2d34c4-c622-11e3-bf01-24c6515278c0',
 '1127b561-ea40-4d5e-95df-daa0a5ebc1e4',
 '1295947c-c623-11e3-bf01-24c6515278c0',
 'fbd899a0-c622-11e3-bf01-24c6515278c0',
 'c6bb32f0-c622-11e3-bf01-24c6515278c0',
 '15895218-c623-11e3-bf01-24c6515278c0',
 '96517058-18d2-4847-8085-768ca8aa1565',
 'b5f90cb8-7304-48fb-a1d3-ff459c7d79d1',
 '532259b8-c622-11e3-bf01-24c6515278c0',
 '54354d24-c622-11e3-bf01-24c6515278c0',
 'c05adc19-2e01-4712-a35a-69eee4d40998',
 '49795e42-c623-11e3-bf01-24c6515278c0',
 '63fef3fe-c622-11e3-bf01-24c6515278c0'])

In [49]:
# just the top 3
top3 = HCC[HCC["sampleID"].isin(['cf2d34c4-c622-11e3-bf01-24c6515278c0',
 '1127b561-ea40-4d5e-95df-daa0a5ebc1e4',
 '1295947c-c623-11e3-bf01-24c6515278c0'])]
print(f"df_len: {len(top3)}")
print(f"Sample Numbers: {top3['sampleID'].nunique()}")
top3.head()

df_len: 73395
Sample Numbers: 3


,chr,start,end,ref,alt,sampleID,Type
15,chr1,758897,758898,C,A,1127b561-ea40-4d5e-95df-daa0a5ebc1e4,Liver-HCC
42,chr1,828366,828367,G,A,1127b561-ea40-4d5e-95df-daa0a5ebc1e4,Liver-HCC
49,chr1,839310,839311,G,T,1127b561-ea40-4d5e-95df-daa0a5ebc1e4,Liver-HCC
114,chr1,1033515,1033516,T,G,cf2d34c4-c622-11e3-bf01-24c6515278c0,Liver-HCC
222,chr1,1298453,1298454,T,G,cf2d34c4-c622-11e3-bf01-24c6515278c0,Liver-HCC


In [50]:
# all 13 outliers
nfia13 = HCC[HCC["sampleID"].isin(nfia_outliers)].copy()
print(f"df_len: {len(nfia13)}")
print(f"Sample Numbers: {nfia13['sampleID'].nunique()}")
nfia13.head()

df_len: 360313
Sample Numbers: 13


,chr,start,end,ref,alt,sampleID,Type
13,chr1,754534,754535,G,A,54354d24-c622-11e3-bf01-24c6515278c0,Liver-HCC
15,chr1,758897,758898,C,A,1127b561-ea40-4d5e-95df-daa0a5ebc1e4,Liver-HCC
42,chr1,828366,828367,G,A,1127b561-ea40-4d5e-95df-daa0a5ebc1e4,Liver-HCC
48,chr1,836966,836967,C,T,c6bb32f0-c622-11e3-bf01-24c6515278c0,Liver-HCC
49,chr1,839310,839311,G,T,1127b561-ea40-4d5e-95df-daa0a5ebc1e4,Liver-HCC


In [51]:
# without top 3

wo_top3 = HCC[~HCC["sampleID"].isin(['cf2d34c4-c622-11e3-bf01-24c6515278c0',
 '1127b561-ea40-4d5e-95df-daa0a5ebc1e4',
 '1295947c-c623-11e3-bf01-24c6515278c0'])]
print(f"df_len: {len(wo_top3)}")
print(f"Sample Numbers: {wo_top3['sampleID'].nunique()}")
wo_top3.head()

df_len: 3771724
Sample Numbers: 311


,chr,start,end,ref,alt,sampleID,Type
0,chr1,10505,10506,C,A,4cff8590-559e-4204-8635-96e11bfeda68,Liver-HCC
1,chr1,20251,20252,A,G,45aedf22-c622-11e3-bf01-24c6515278c0,Liver-HCC
2,chr1,77903,77904,A,G,e34bcc5e-c622-11e3-bf01-24c6515278c0,Liver-HCC
3,chr1,94993,94994,T,C,221d0290-c623-11e3-bf01-24c6515278c0,Liver-HCC
4,chr1,121080,121081,T,C,19fe27d8-c623-11e3-bf01-24c6515278c0,Liver-HCC


In [52]:
# without nfia 13 outliers

wo_nfia13 = HCC[~HCC["sampleID"].isin(nfia_outliers)].copy()
print(f"df_len: {len(wo_nfia13)}")
print(f"Sample Numbers: {wo_nfia13['sampleID'].nunique()}")
wo_nfia13.head()

df_len: 3484806
Sample Numbers: 301


,chr,start,end,ref,alt,sampleID,Type
0,chr1,10505,10506,C,A,4cff8590-559e-4204-8635-96e11bfeda68,Liver-HCC
1,chr1,20251,20252,A,G,45aedf22-c622-11e3-bf01-24c6515278c0,Liver-HCC
2,chr1,77903,77904,A,G,e34bcc5e-c622-11e3-bf01-24c6515278c0,Liver-HCC
3,chr1,94993,94994,T,C,221d0290-c623-11e3-bf01-24c6515278c0,Liver-HCC
4,chr1,121080,121081,T,C,19fe27d8-c623-11e3-bf01-24c6515278c0,Liver-HCC


#### creating a function to get the alterante tri nt sequence

In [53]:
def get_tri(row):
    return bgreference.hg19(row['chr'], row['start'], 3)

In [54]:
def compute_tri_alt(x):
    """
    Compute the alternate triplet sequence from a DataFrame row or a dict 
    with the reference triplet and the alternated nucleotide
    """
    my_alt = x['alt']
    my_tri = x['tri']

    my_alt_tri = my_tri[0] + my_alt + my_tri[2]
    
    return my_alt_tri

In [55]:
def create_tri_dict():
    """
    Construct a dictionary where the values are the 192 combinations 
    that result of all possible changes of any nucleotide taking into account its context
    """
    
    nucleotides = ['A', 'C', 'T', 'G']

    tri_nucleotides = list()

    for my_nucleotide1 in nucleotides:
        for my_nucleotide2 in nucleotides:
            for my_nucleotide3 in nucleotides:
                my_tri = [my_nucleotide1,my_nucleotide2,my_nucleotide3]

                tri_nucleotides.append(my_tri)

    trinucleotides_dict = dict()

    for my_tri in tri_nucleotides:
        my_ref_tri = my_tri[0] + my_tri[1] + my_tri[2]
        mid_nucl = my_tri[1]

        for my_nucleotide in nucleotides:
            if my_nucleotide != mid_nucl:
                my_alt_tri = my_tri[0] + my_nucleotide + my_tri[2]

                trinucleotides_dict[(my_ref_tri, my_alt_tri)] = 0
                
    return trinucleotides_dict

#### Tagging each of the df
so that we can run it all together while seperatig the outlier status

In [ ]:
HCC["status"] = "PCAWG_all_samples"
HCC_outliers["status"] = "PCAWG_8_outliers"
HCC_wo_outliers["status"] = "PCAWG_wo8_outliers"  #without 8 outliers

#nfia
top3["status"] = "nfia_top3_outliers"
nfia13["status"] = "nfia_13_outliers"
wo_top3["status"] = "nfia_wotop3_outliers" #without top 3 outliers
wo_nfia13["status"] = "nfia_wo13_outliers" #without 13 outliers


In [57]:
all_df = pd.concat([HCC, HCC_outliers, HCC_wo_outliers, top3, nfia13, wo_top3, wo_nfia13])
all_df.head()

,chr,start,end,ref,alt,sampleID,Type,status
0,chr1,10505,10506,C,A,4cff8590-559e-4204-8635-96e11bfeda68,Liver-HCC,PCAWG_all_samples
1,chr1,20251,20252,A,G,45aedf22-c622-11e3-bf01-24c6515278c0,Liver-HCC,PCAWG_all_samples
2,chr1,77903,77904,A,G,e34bcc5e-c622-11e3-bf01-24c6515278c0,Liver-HCC,PCAWG_all_samples
3,chr1,94993,94994,T,C,221d0290-c623-11e3-bf01-24c6515278c0,Liver-HCC,PCAWG_all_samples
4,chr1,121080,121081,T,C,19fe27d8-c623-11e3-bf01-24c6515278c0,Liver-HCC,PCAWG_all_samples


In [59]:
all_df["status"].unique()

array(['PCAWG_all_samples', 'PCAWG_8_outliers', 'PCAWG_wo8_outliers',
       'nfia_top3_outliers', 'nfia_13_outliers', 'nfia_wotop3_outliers',
       'nfia_wo13_outliers'], dtype=object)

#### Computing mutation Probabilities

In [60]:
my_probs_df = pd.DataFrame()
my_probs_df['mutation'] = list(create_tri_dict().keys())

for sample_type in all_df['status'].unique().tolist():
    
    # read mutation file
    mutations = all_df[all_df["status"] == sample_type]
        
    print(sample_type)
    
    # get the sampleids corresponding to the tumor type
    tmp_samples_pcawg = list(all_df[(all_df['status']==sample_type)]['sampleID'])
        
    # subset mutations
    subset = mutations[mutations['sampleID'].isin(tmp_samples_pcawg)]

    # generate tri-nucleotide context
    subset['tri']=subset.apply(get_tri, axis=1)

    subset['alt_tri'] = subset.apply(lambda x: compute_tri_alt(x), axis=1)


    # remove line if it has 'N' nucleotide in ref or mut trinucleotide
    subset = subset[~subset['tri'].str.contains('N')]
    subset = subset[~subset['alt_tri'].str.contains('N')]

    subset['mutation'] = list(zip(subset.tri, subset.alt_tri))

    trinucleotides_dict = create_tri_dict()

    # Compute the counts of each triplet alteration
    for my_count in subset['mutation'].value_counts().iteritems():
        #if my_count[0] in subset['mutation'].tolist():
        trinucleotides_dict[my_count[0]] = my_count[1]

    # Divide each count by the counts of the reference trinucleotide
    for my_tri in trinucleotides_dict.keys():
        my_ref_tri = my_tri[0]
        #if 'N' in my_ref_tri:
        #    continue
        total_count = tri_freq_df[tri_freq_df['tri'] == my_ref_tri]['count'].values[0]

        trinucleotides_dict[my_tri] = trinucleotides_dict[my_tri]/total_count

    # Normalize the values
    all_values_sum = sum(trinucleotides_dict.values())
    for my_tri in trinucleotides_dict.keys():
        trinucleotides_dict[my_tri] = trinucleotides_dict[my_tri]/all_values_sum

    column_name = 'Probability_' + sample_type

    probs_df = pd.DataFrame({'mutation' : list(trinucleotides_dict.keys()) , column_name : list(trinucleotides_dict.values()) })
    my_probs_df = pd.merge(my_probs_df, probs_df, on='mutation')


PCAWG_all_samples


/tmp/ipykernel_4138474/1767811549.py:32: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for my_count in subset['mutation'].value_counts().iteritems():


PCAWG_8_outliers


/tmp/ipykernel_4138474/1767811549.py:32: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for my_count in subset['mutation'].value_counts().iteritems():


PCAWG_wo8_outliers


...

/tmp/ipykernel_4138474/1767811549.py:32: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for my_count in subset['mutation'].value_counts().iteritems():


nfia_wotop3_outliers


/tmp/ipykernel_4138474/1767811549.py:32: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for my_count in subset['mutation'].value_counts().iteritems():


nfia_wo13_outliers


/tmp/ipykernel_4138474/1767811549.py:32: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for my_count in subset['mutation'].value_counts().iteritems():


In [61]:
my_probs_df

,mutation,Probability_PCAWG_all_samples,Probability_PCAWG_8_outliers,Probability_PCAWG_wo8_outliers,Probability_nfia_top3_outliers,Probability_nfia_13_outliers,Probability_nfia_wotop3_outliers,Probability_nfia_wo13_outliers
0,"(AAA, ACA)",0.001757,0.001610,0.001770,0.007259,0.002493,0.001662,0.001677
1,"(AAA, ATA)",0.002032,0.002242,0.002014,0.002116,0.001859,0.002030,0.002051
2,"(AAA, AGA)",0.003734,0.003977,0.003713,0.003169,0.003882,0.003744,0.003718
3,"(AAC, ACC)",0.001318,0.001225,0.001326,0.004118,0.001778,0.001269,0.001268
4,"(AAC, ATC)",0.002091,0.002279,0.002076,0.002471,0.001962,0.002085,0.002106
...,...,...,...,...,...,...,...,...
187,"(GGT, GCT)",0.002026,0.001533,0.002068,0.002387,0.001750,0.002020,0.002056
188,"(GGT, GTT)",0.007413,0.005758,0.007553,0.006250,0.006404,0.007433,0.007523
189,"(GGG, GAG)",0.006925,0.005792,0.007021,0.006164,0.006141,0.006938,0.007010
190,"(GGG, GCG)",0.002002,0.001438,0.002050,0.002882,0.001805,0.001987,0.002023


##### some checks to make sure the code has been run properly

In [63]:
#Check1
my_probs_df.sum()

mutation                            (AAA, ACA, AAA, ATA, AAA, AGA, AAC, ACC, AAC, ...
Probability_PCAWG_all_samples                                                     1.0
Probability_PCAWG_8_outliers                                                      1.0
Probability_PCAWG_wo8_outliers                                                    1.0
Probability_nfia_top3_outliers                                                    1.0
Probability_nfia_13_outliers                                                      1.0
Probability_nfia_wotop3_outliers                                                  1.0
Probability_nfia_wo13_outliers                                                    1.0
dtype: object

In [66]:
pbt.cleanup()

In [ ]:
#my_probs_df.to_csv(OUTPUT_DIR / "PCAWG_trinucleotide_mutation_probabilities.csv",sep="\t",index=False)